In [7]:
import pandas as pd
import json
from api_caller import call_api

In [8]:
df = pd.read_csv("data/EuroParl/preprocessed/2009filtered.csv")
df_subset = df.iloc[2000:2500].copy()

In [9]:
def classify_batch(texts):
    joined_text = "\n\n".join(
        [f"ID {i}: {text}" for i, text in enumerate(texts)]
    )

    user_prompt = f"""You are a political ideology detection system.

Your task is to detect STRONG ideological value expression.

A speech should receive a high score ONLY IF it clearly expresses
a political principle or ideological position that could be mapped
onto a political survey (e.g., redistribution, EU integration,
national sovereignty, migration, social equality, market regulation,
democracy, rule of law, etc.).

STRICT CRITERIA:

The speech MUST:
- Advocate or oppose a political principle
- Express how society, the EU, or government SHOULD be structured
- Reveal a stable ideological commitment attributable to the speaker or their party

The speech must NOT be classified as value-expressing if it:
- Expresses generic hope or praise
- Uses polite or diplomatic language
- Evaluates events without ideological reasoning
- Contains general positive or negative sentiment only

IMPORTANT:
Generic approval is NOT ideological.

Be extremely conservative.
If the speech does not clearly state a political principle,
assign a score below 0.2.

Scoring guide:
0.0-0.2 → no ideological value content
0.3-0.5 → weak or vague value signals
0.6-0.8 → clear ideological positioning
0.9-1.0 → strong, explicit ideological commitment

Return STRICT JSON in the same order:
[
  {{
    "score": 0.0-1.0,
    "reason": "..."
  }}
]

IMPORTANT: Always return valid JSON array with exactly {len(texts)} entries, 
even if the score is 0.0 for some items.

Texts:
{joined_text}

"""

    response = call_api(user_prompt)

    return json.loads(response["choices"][0]["message"]["content"])

In [10]:
# Process in chunks for speed
batch_size = 10
indices = list(range(0, len(df_subset), batch_size))
results = []

for i in range(0, len(df_subset), batch_size):
    print(f"Progress: {i/len(df_subset)*100}%")
    batch_texts = df_subset["en"].iloc[i:i+batch_size].tolist()
    batch_results = classify_batch(batch_texts)
    results.extend(batch_results)

Progress: 0.0%


In [ ]:
df_subset["reason"] = [r["reason"] for r in results]
df_subset["score"] = [r["score"] for r in results]
final_df = df_subset[["en", "score", "reason", "EU Party"]]
final_df.to_csv(
    "data/EuroParl/scored_gpt/scored_data_new.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)


In [ ]:
len(results)

1000